# 05. Annotation Mask Test

This notebook tests the annotation mask module on the sample pose CSV.

The annotation step adds metadata columns to the pose dataframe without removing any frames:

- `use_for_analysis`
- `segment_type`
- `set_id`
- `rep_id`
- `phase`

This notebook assumes that the previous checks are already working:

- 00_environment_check
- 01_data_loading_test
- 02_validation_test
- 03_raw_visualization_test
- 04_normalization_test

Three scenarios are tested:

1. No annotation file provided — full-sequence fallback
2. Annotation file loaded and validated
3. Annotation applied — metadata columns added to normalized dataframe

An overlap detection edge case is also verified.

In [ ]:
import json

import pandas as pd

from movement.annotation import (
    ANNOTATION_OUTPUT_COLUMNS,
    apply_annotation,
    load_annotation_csv,
    validate_annotation,
)
from movement.config import LANDMARKS
from movement.io import load_pose_csv
from movement.normalization import normalize_pose_by_hip_torso


In [ ]:
csv_path = "../data/sample/mediapipe_forward_bend_sample.csv"
ann_path = "../data/sample/mediapipe_forward_bend_sample_annotation.csv"

df = load_pose_csv(csv_path)
norm_df, _ = normalize_pose_by_hip_torso(df=df, landmarks=LANDMARKS)

print(f"pose dataframe : {norm_df.shape[0]} frames, {norm_df.shape[1]} columns")


## Case 1: No Annotation Provided (Full-Sequence Fallback)

When `ann_df=None`, all frames are marked `use_for_analysis=True` and `segment_type='full_sequence'`.

This allows the pipeline to run without any manual annotation file.

In [ ]:
annotated_fallback, report_fallback = apply_annotation(norm_df, ann_df=None)

print(json.dumps(report_fallback, indent=2))


In [ ]:
print("annotation columns added:", ANNOTATION_OUTPUT_COLUMNS)
print()
print("use_for_analysis unique:", annotated_fallback["use_for_analysis"].unique().tolist())
print("segment_type unique:    ", annotated_fallback["segment_type"].unique().tolist())
print("set_id unique:          ", annotated_fallback["set_id"].unique().tolist())
print("rep_id unique:          ", annotated_fallback["rep_id"].unique().tolist())


## Case 2: Load and Validate Annotation File

The annotation CSV follows the naming convention: `{pose_csv_stem}_annotation.csv` in the same directory.

Required columns: `segment_type`, `set_id`, `rep_id`, `start_frame`, `end_frame`, `use_for_analysis`

In [ ]:
ann_df = load_annotation_csv(ann_path)

ann_df


In [ ]:
val_report = validate_annotation(ann_df, norm_df)

print(json.dumps(val_report, indent=2))


## Case 3: Apply Annotation

Annotation is applied by adding metadata columns to the normalized pose dataframe.

All frames start with `use_for_analysis=False`.
Frames inside annotated segments are updated according to the annotation file.
Frames outside annotated ranges remain excluded.

In [ ]:
annotated_df, ann_report = apply_annotation(norm_df, ann_df)

safe_report = {k: v for k, v in ann_report.items() if k != "validation"}
print(json.dumps(safe_report, indent=2))


In [ ]:
print("=== segment_type counts ===")
print(annotated_df["segment_type"].value_counts().to_string())
print()
print("=== use_for_analysis counts ===")
print(annotated_df["use_for_analysis"].value_counts().to_string())


In [ ]:
rep_frames = annotated_df[annotated_df["segment_type"] == "rep"][
    ["frame", "timestamp", "use_for_analysis", "segment_type", "set_id", "rep_id"]
]

print(f"rep frames: {len(rep_frames)} total")
print()
print(
    rep_frames
    .groupby(["set_id", "rep_id"])
    .agg(start=("frame", "min"), end=("frame", "max"), count=("frame", "count"))
    .to_string()
)


## Edge Case: Overlap Detection

Overlapping annotation ranges should not be silently accepted.

`validate_annotation` must return `passed=False` and report the conflicting row pairs.
`apply_annotation` must raise a `ValueError`.

In [ ]:
bad_ann = pd.DataFrame({
    "segment_type": ["rep", "rep"],
    "set_id": pd.array([1, 1], dtype="Int64"),
    "rep_id": pd.array([1, 2], dtype="Int64"),
    "start_frame": [10, 50],
    "end_frame":   [70, 90],
    "use_for_analysis": [True, True],
})

overlap_report = validate_annotation(bad_ann, norm_df)
print("passed:            ", overlap_report["passed"])
print("overlapping_pairs: ", overlap_report["overlapping_pairs"])
print()

try:
    apply_annotation(norm_df, bad_ann)
except ValueError as e:
    print("ValueError raised as expected.")


## Interpretation

Expected results for each case:

**Case 1 — Full-sequence fallback:**

- `annotation_provided` is `False`
- all 120 frames have `use_for_analysis=True`
- `segment_type` is `'full_sequence'` for all frames

**Case 2 — Validation:**

- `validate_annotation` returns `passed=True`
- `overlapping_pairs`, `out_of_bounds_rows`, `invalid_range_rows` are all empty
- `unknown_segment_types` is empty

**Case 3 — Annotation applied:**

- `annotation_provided` is `True`
- `num_analysis_frames + num_excluded_frames == num_total_frames`
- rep frames group by `set_id` and `rep_id` correctly
- original `frame` column is unchanged

**Edge case — Overlap:**

- `validate_annotation` returns `passed=False`
- `overlapping_pairs` contains the conflicting row indices
- `apply_annotation` raises `ValueError`

Note: the frame boundaries in the sample annotation file are approximate.
Adjust `start_frame` and `end_frame` values after visual inspection of the pose sequence.